# Harmonization Benchmark

Systematic evaluation of **25 batch-correction methods** across **10 sample-removal strategies** for the multi-platform B-cell lymphoma transcriptomics dataset.  
Full 4D cross-product: 10 strategies × 4 imputation approaches × 25 methods × 2 post-removal options = **2,000 combinations**.  
SOM (oposSOM) run for top `TOP_K_SOM` combinations ranked by pre-SOM batch R².

| Section | Scope |
|---|---|
| 0 | Setup: imports, `BIO_COL`/`BATCH_COL` global config, palettes, PALETTE_GROUPS, plot helpers |
| 1 | Data loading and baseline batch R²; `Diagnosis_with_coo` / `Diagnosis_unified_with_coo` derived columns |
| 2 | 10 sample-removal strategies (S0, A, B, C, D, E1–E3, F, G) + gene filtering (strict and imputed) |
| 3 | 18 harmonization method functions (low → medium → high harshness) |
| 4 | Full 4D cross-product evaluation: 10 × 4 × 25 × 2 = 2,000 combinations, heatmaps, top-K selection |
| 5 | SOM analysis: top `TOP_K_SOM` strategy+imputation+method+post-removal combinations |
| 6 | Post-SOM evaluation: batch R², embeddings, NN batch dominance test |
| 7 | SOM portrait comparison grids |
| 8 | Quantitative summary table + interpretation |


## 0 — Setup


In [ ]:
# ── sys.path — add harmonization-scripts to the search path ────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'harmonization-scripts'))

# ── Standard ─────────────────────────────────────────────────────────────────
import warnings, pickle, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import seaborn as sns
from tqdm.notebook import tqdm
warnings.filterwarnings("ignore")

# ── Dimensionality reduction ──────────────────────────────────────────────────
from sklearn.manifold import TSNE
from umap import UMAP

# ── R bridge ──────────────────────────────────────────────────────────────────
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
pandas2ri.activate()

# ── sklearn ───────────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# ── Global plot config ────────────────────────────────────────────────────────
plt.rcParams.update({"pdf.fonttype": "truetype", "svg.fonttype": "none", "figure.dpi": 150})
sns.set_style("ticks")

# ── Paths ─────────────────────────────────────────────────────────────────────
REMOTE_ROOT   = "$FL_DATA_ROOT"
EXP_PATH      = f"{REMOTE_ROOT}/comb_exp.tsv"
ANN_PATH      = "comb_ann_unified.csv"
FSQN_R_PATH   = f"{REMOTE_ROOT}/fsqn_normalized_exp_R.tsv"
ANN_NORM_PATH = f"{REMOTE_ROOT}/ann_normalized_fsqn_R.tsv"
OUT_DIR       = "harmonization_results"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Global annotation column config — change here to affect the whole notebook ──
BIO_COL   = "Diagnosis_cell_type_unified"   # main biology grouping column
BATCH_COL = "RNA_BATCH"                     # batch variable for metrics and correction
TOP_K_SOM = 15                              # top strategy+method combinations for SOM
# ── Shared utilities from harmonization-scripts/ ─────────────────────────────
from bench_shared import (
    RARE_GROUPS, NORMAL_GROUPS,
    BAD_BATCHES_A, BAD_COHORTS_A, BAD_BATCHES_B, BAD_COHORTS_B,
    log_transform_by_cohort,
    pca_variance_explained_by_batch,
    r2_batch,
    apply_filter,
    identify_outlier_batches,
    prepare_dataset,
    prepare_dataset_imputed,
    build_filter_strategies,
    s3_key_exp, s3_key_metrics, s3_exists,
    upload_exp_to_s3, download_exp_from_s3,
    METHODS, HARSHNESS_COLORS,
)
from load_cross_product_results import load_metrics, load_exp, load_top_k, cross_results


In [ ]:
??log_transform_by_cohort

In [ ]:
# ── Custom dimensionality reduction plot helpers ──────────────────────────────
import math as _math

def _lookup_color(palette: dict, g) -> str:
    """Dict lookup that handles float(nan) and None keys."""
    if g in palette:
        return palette[g]
    if isinstance(g, float) and _math.isnan(g):
        for k in palette:
            if isinstance(k, float) and _math.isnan(k):
                return palette[k]
    return palette.get(str(g), "#888888")


def _draw_grouped_legend(ax, unique_groups, colors: dict, palette_groups: dict) -> None:
    """Render legend with bold group subtitles and visual separators."""
    from matplotlib.lines import Line2D
    handles, labels = [], []
    shown = set()
    for group_name, group_keys in palette_groups.items():
        present = [k for k in group_keys if k in [str(g) for g in unique_groups]]
        if not present:
            continue
        handles.append(Line2D([0], [0], color="none"))
        labels.append(f"── {group_name} ──")
        for k in present:
            c = _lookup_color(colors, k)
            handles.append(Line2D([0], [0], marker="o", color="none",
                                  markerfacecolor=c, markersize=5))
            labels.append(k)
            shown.add(k)
    for g in unique_groups:
        if str(g) not in shown:
            c = _lookup_color(colors, g)
            handles.append(Line2D([0], [0], marker="o", color="none",
                                  markerfacecolor=c, markersize=5))
            labels.append(str(g))
    ax.legend(handles, labels, fontsize=5, markerscale=1,
              bbox_to_anchor=(1.01, 1), loc="upper left")


def _scatter_grouped(coords, grouping, palette, legend, ax, title,
                     palette_groups=None):
    ax = ax or plt.gca()
    groups = grouping.values
    unique = pd.unique(groups)
    colors = (palette if isinstance(palette, dict)
              else dict(zip(unique, sns.color_palette("tab20", len(unique)))))
    for g in unique:
        mask = groups == g
        c    = _lookup_color(colors, g) if isinstance(colors, dict) else "#888888"
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[c], s=4, alpha=0.6, label=str(g), linewidths=0)
    if legend != "off" and legend is not False:
        if palette_groups and isinstance(colors, dict):
            _draw_grouped_legend(ax, unique, colors, palette_groups)
        else:
            ax.legend(fontsize=5, markerscale=2,
                      bbox_to_anchor=(1.01, 1), loc="upper left")
    if title:
        ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


def pca_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
             n_components=2, palette_groups=None):
    X   = StandardScaler().fit_transform(exp_df.fillna(0).values)
    pca = PCA(n_components=n_components, random_state=42)
    coords   = pca.fit_transform(X)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    ax = _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=7)
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=7)
    return ax


def umap_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
              n_neighbors=15, min_dist=0.3, palette_groups=None):
    X      = StandardScaler().fit_transform(exp_df.fillna(0).values)
    coords = UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                  random_state=42).fit_transform(X)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    return _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)


def tsne_plot(exp_df, grouping, palette=None, legend=None, ax=None, title=None,
              perplexity=30, palette_groups=None):
    X      = StandardScaler().fit_transform(exp_df.fillna(0).values)
    n_comp = min(50, X.shape[1] - 1, X.shape[0] - 1)
    X_pca  = PCA(n_components=n_comp, random_state=42).fit_transform(X)
    coords = TSNE(n_components=2, perplexity=min(perplexity, len(X) - 1),
                  random_state=42, init="pca").fit_transform(X_pca)
    grouping = grouping.reindex(exp_df.index).fillna("NA")
    return _scatter_grouped(coords, grouping, palette, legend, ax, title, palette_groups)


print("Plot helpers defined.")


In [ ]:
# ── Color palettes ────────────────────────────────────────────────────────────
# Adapt these from all_cohorts_assembly.ipynb or define project-specific ones.
# Defaults: seaborn tab20 / husl for unlabelled categories.

# RNA_BATCH palette — one color per sequencing platform/batch category
# Update with the full list from comb_ann_unified.csv if needed
lymphoma_ontogeny_palette = {
    # --- NORMAL B-CELLS: THE COLD SPECTRUM (Blues, Teals, Greens) ---
    # Primary/Early Development
    'Bone_marrow_CD19+': 'blue',    # Deepest Navy
    'Immature': '#08519c',             # Dark Blue
    'Naive': '#4292c6',                # Medium Blue
    
    # Germinal Center (Cold variants)
    'GC': '#006d2c',                   # Dark Green
    'Centroblast': '#41ab5d',          # Medium Green
    'Centrocyte': '#a1d99b',           # Pale Green
    
    # Mature & Memory
    'B_cells': 'lawngreen',              # Forest Green
    'MZ': 'olive',                   # Seafoam Green
    'Memory': '#00ced1',               # Dark Turquoise
    
    # Terminal Differentiation (Cold Neutrals)
    'Plasmablast': '#5f9ea0',          # Cadet Blue
    'Plasma': 'grey',               # Steel Blue-Gray

    # --- MALIGNANT TYPES: THE WARM SPECTRUM (Reds, Oranges, Purples, Black) ---
    # Follicular (The Purple/Magenta family)
    'Follicular_Lymphoma': '#6a0dad',      # Vivid Purple
    'Follicular_Lymphoma_GCB': '#9400d3',   # Dark Violet
    
    # DLBCL (The Red/Orange family)
    'Diffuse_Large_B_Cell_Lymphoma_ABC': '#ff4500', # Orange-Red
    'Diffuse_Large_B_Cell_Lymphoma_GCB': '#d62728', # Classic Red
    'Diffuse_Large_B_Cell_Lymphoma': '#8b0000',    # Dark Red
    
    # High-Grade & Burkitt (Deep Contrast)
    'High_Grade_B_Cell_Lymphoma': '#ff00ff',       # Magenta/Fuchsia
    'Burkitt_Lymphoma': 'orange'                  # Absolute Black
}

rna_batch_palette = {
    # RNASeq Group (Cooler tones: Blues, Purples, Teals, Greens)
    "RNASeq_FF_Total": "#1f77b4",  # Deep Blue
    "RNASeq_FF_PolyA": "#17becf",  # Bright Cyan
    "RNASeq_FFPE_Exome_capture": "#9467bd",  # Muted Purple
    "RNASeq_FF_rRNADepletion": "#000080",  # Navy
    "RNASeq_FFPE_PolyA": "#8c564b",  # Deep Brown-ish Purple
    "RNASeq_FF_Unknown": "#bcbd22",  # Olive Green
    "RNASeq_FF_Exome_capture": "#2ca02c",  # Forest Green
    # Affymetrix Family (Reds, Oranges, Pinks)
    "GPL96+97_FF_Unknown": "#d62728",  # Bright Red
    "GPL570_FF_Unknown": "#ff7f0e",  # Vivid Orange
    "GPL570_Unknown_Unknown": "#ffbb78",  # Pale Orange
    "GPL570_FFPE_Unknown": "#dbdb8d",  # Pale Olive
    "GPL96_FF_Unknown": "#f7b6d2",  # Pale Pink
    "GPL96_FFPE_Unknown": "#e377c2",  # Pink
    "GPL6244_FF_Unknown": "#e9967a",  # Dark Salmon
    "GPL6244_FFPE_Unknown": "#fa8072",  # Salmon
    "GPL13938_FFPE_Unknown": "#800000",  # Maroon
    # Agilent Family (Blues, Teals, Lavenders, Grays)
    "GPL1708_FF_Unknown": "#7f7f7f",  # Medium Gray
    "GPL14951_FFPE_Unknown": "#c49c94",  # Pale Brown
    "GPL17077_FF_Unknown": "#c5b0d5",  # Pale Lavender
    "GPL13158_FF_Unknown": "#ad494a",  # Dusty Red
    "GPL17586_FF_Unknown": "#008b8b",  # Dark Cyan
    "GPL17047_FF_Unknown": "#4682b4",  # Steel Blue
    "GPL887_FFPE_Unknown": "#b0c4de",  # Light Steel Blue
    # Illumina Family (Greens)
    "GPL10739_FF_Unknown": "#9edae5",  # Pale Blue (Illumina original)
    "GPL8432_FFPE_Unknown": "#006400",  # Dark Green
    "GPL20188_FF_Unknown": "#556b2f",  # Dark Olive Green
    "GPL23541_FF_Unknown": "#8fbc8f",  # Dark Sea Green
    # Other / Miscellaneous (Neutrals)
    "GPL16686_FF_Unknown": "#333333",  # Near Black
    "GPL26356_FF_Unknown": "#a0522d",  # Sienna
}
cohort_palette = {
    "PUB_DLBCL_CARTFARAMAND": "#f87d06",
    "PUB_DLBCL_JIANGGSE147986": "#1482b7",
    "PUB_DLBCL_GOYA": "#b72114",
    "PUB_DLBCL_NCICCR": "#b7b014",
    "PUB_DLBCL_PARKSRP100394": "#5bd66f",
    "PUB_DLBCL_SHISRP312586": "#2306f8",
    "PUB_DLBCL_YANCART": "#d67e5b",
    "PUB_FL_DAVE": "#0626f8",
    "PUB_FL_GSE62241": "#b71481",
    "PUB_FL_HUET": "#14b75d",
    "PUB_FL_PARSA": "#146db7",
    "PUB_FL_GUO": "#a3d65b",
    "PUB_FL_GSE148070": "#5b95d6",
    "PUB_FL_BRODTKORB": "#3514b7",
    "PUB_FL_CIBERSORTX": "#9ff806",
    "PUB_FL_HORN": "#5bd6cb",
    "PUB_FL_PASTORE": "#d66f5b",
    "PUB_FL_LEBRUN": "#b71496",
    "PUB_FL_ORICCHIO": "#9d5bd6",
    "PUB_FL_TASKINEN": "#14b772",
    "PUB_FL_DREYLING": "#8d5bd6",
    "PUB_FL_NCIFFPEREBUILT": "#f80625",
    "PUB_FL_NCISTAUDT": "#f80682",
    "PUB_FL_TOBIN": "#069ef8",
    "PUB_DLBCL_GSE11318": "#f80644",
    "PUB_DLBCL_BAGS": "#5bd6ac",
    "PUB_DLBCL_ETABM346": "#5bd660",
    "PUB_DLBCL_DLC1": "#5bd6bb",
    "PUB_DLBCL_NCIACALA": "#06f860",
    "PUB_DLBCL_GSE10846": "#5b85d6",
    "PUB_DLBCL_GSE117556": "#4914b7",
    "PUB_DLBCL_GSE19246": "#9b06f8",
    "PUB_DLBCL_GSE22898": "#c2d65b",
    "PUB_DLBCL_GSE23501": "#bef806",
    "PUB_DLBCL_GSE31312": "#b73614",
    "PUB_DLBCL_GSE32918": "#f706f8",
    "PUB_DLBCL_GSE34171": "#5e14b7",
    "PUB_DLBCL_GSE64555": "#d65bb3",
    "PUB_DLBCL_GSE87371": "#06dbf8",
    "PUB_DLBCL_GSE93984": "#d65ba3",
    "PUB_DLBCL_LYMPHOMICS": "#d65b84",
    "COLL_DLBCL_UMLOSSOSTRANSL": "#5b75d6",
    "CRON3": "#06f8f6",
    "E-MTAB-3974": "#f806da",
    "GSE107683": "#5bd1d6",
    "GSE12195": "#f89c06",
    "GSE12366": "#0607f8",
    "GSE12453": "#b2d65b",
    "GSE12845": "#66d65b",
    "GSE13411": "#14b787",
    "GSE136248": "#4206f8",
    "GSE136249": "#1bb714",
    "GSE141005": "#b906f8",
    "GSE144089": "#f80663",
    "GSE149089": "#f85e06",
    "GSE15271": "#aab714",
    "GSE156751": "#14b748",
    "GSE17186": "#1458b7",
    "GSE19599": "#b78614",
    "GSE22886": "#d806f8",
    "GSE36907": "#bc5bd6",
    "GSE38697": "#b79b14",
    "GSE46261": "#45b714",
    "GSE51528": "#1444b7",
    "GSE55267": "#1497b7",
    "GSE56314": "#b714ab",
    "GSE61912": "#76d65b",
    "GSE64028": "#06f841",
    "GSE68878": "#d68e5b",
    "GSE69033": "#06f822",
    "GSE85543": "#d2d65b",
    "GSE89282": "#06bdf8",
    "GSE99635": "#30b714",
    "Leandro-Normals": "#af14b7",
    "PUB_Beguelin_GSE45982": "#d6ca5b",
    "PUB_Health_B_cells_GSE110669": "#d69e5b",
    "PUB_Health_B_cells_GSE111310": "#b7141c",
    "PUB_Health_B_cells_GSE115655": "#b71431",
    "PUB_Health_B_cells_GSE119234": "#06f87f",
    "PUB_Health_B_cells_GSE139833": "#59b714",
    "PUB_Health_B_cells_GSE154583": "#5b66d6",
    "PUB_Health_B_cells_GSE92387": "#f8bb06",
    "PUB_Health_Bunting_GSE84022": "#d65b94",
    "PUB_Pastore_GSE119103": "#ac5bd6",
    "PUB_Suntsova_GSE120795": "#80f806",
    "Kassandra": "#14b79c",
    "SOM": "#61f806",
    "MDA_Strati_FL": "#2014b7",
}

platform_palette = {
    # Standalone / Sequencing (Indigo/Purple Spectrum)
    "RNASeq": "#4B0082",  # Indigo
    "RNAseq": "#4B0082",  # Matches RNASeq
    "Kassandra": "#6A5ACD",  # SlateBlue (Deconvolution/Virtual)
    # Affymetrix Group (Oranges & Reds)
    "GPL96+97": "#FF4500",  # OrangeRed
    "GPL96+GPL97": "#FF4500",  # Synonymous with GPL96+97
    "GPL570": "#FF8C00",  # DarkOrange
    "GPL96": "#B22222",  # FireBrick
    "GPL6244": "#E9967A",  # DarkSalmon (HuGene 1.0 ST)
    "GPL13938": "#CD5C5C",  # IndianRed
    "A-GEOD-19803": "#A52A2A",  # Brown/Dark Red (Affy Gene 1.0 ST)
    # Agilent Group (Blues & Teals)
    "GPL1708": "#008080",  # Teal
    "GPL14951": "#4682B4",  # SteelBlue
    "GPL17077": "#00CED1",  # DarkTurquoise
    "GSE17077": "#00CED1",  # Matches GPL17077 (likely a typo in source)
    "GPL13158": "#1E90FF",  # DodgerBlue
    "GPL17586": "#20B2AA",  # LightSeaGreen
    "GPL17047": "#5F9EA0",  # CadetBlue
    "GPL887": "#B0C4DE",  # LightSteelBlue
    # Illumina Group (Greens)
    "GPL10739": "#2E8B57",  # SeaGreen
    "GPL10739+GPL19251": "#3CB371",  # MediumSeaGreen
    "GPL20188": "#6B8E23",  # OliveDrab
    "GPL23541": "#8FBC8F",  # DarkSeaGreen
    "GPL8432": "#006400",  # DarkGreen
    # Other / Miscellaneous (Earth Tones & Grays)
    "GPL16686": "#8B4513",  # SaddleBrown
    "GPL26356": "#A0522D",  # Sienna
    float("nan"): "#D3D3D3",  # LightGray (for missing values)
    None: "#D3D3D3",  # Safety for None types
}

In [ ]:
# ── PALETTE_GROUPS — defines legend groupings for _draw_grouped_legend ────────
PALETTE_GROUPS = {
    "rna_batch_palette": {
        "RNASeq Group": [
            "RNASeq_FF_Total", "RNASeq_FF_PolyA", "RNASeq_FFPE_Exome_capture",
            "RNASeq_FF_rRNADepletion", "RNASeq_FFPE_PolyA", "RNASeq_FF_Unknown",
            "RNASeq_FF_Exome_capture",
        ],
        "Affymetrix Family": [
            "GPL96+97_FF_Unknown", "GPL570_FF_Unknown", "GPL570_Unknown_Unknown",
            "GPL570_FFPE_Unknown", "GPL96_FF_Unknown", "GPL96_FFPE_Unknown",
            "GPL6244_FF_Unknown", "GPL6244_FFPE_Unknown", "GPL13938_FFPE_Unknown",
        ],
        "Agilent Family": [
            "GPL1708_FF_Unknown", "GPL14951_FFPE_Unknown", "GPL17077_FF_Unknown",
            "GPL13158_FF_Unknown", "GPL17586_FF_Unknown", "GPL17047_FF_Unknown",
            "GPL887_FFPE_Unknown",
        ],
        "Illumina Family": [
            "GPL10739_FF_Unknown", "GPL8432_FFPE_Unknown", "GPL20188_FF_Unknown",
            "GPL23541_FF_Unknown",
        ],
        "Other": ["GPL16686_FF_Unknown", "GPL26356_FF_Unknown"],
    },
    "lymphoma_ontogeny_palette": {
        "Normal B Cells": [
            "Bone_marrow_CD19+", "Immature", "Naive", "GC", "Centroblast",
            "Centrocyte", "B_cells", "MZ", "Memory", "Plasmablast", "Plasma",
        ],
        "Malignant Types": [
            "Follicular_Lymphoma", "Follicular_Lymphoma_GCB",
            "Diffuse_Large_B_Cell_Lymphoma_ABC", "Diffuse_Large_B_Cell_Lymphoma_GCB",
            "Diffuse_Large_B_Cell_Lymphoma", "High_Grade_B_Cell_Lymphoma",
            "Burkitt_Lymphoma",
        ],
    },
}

print("PALETTE_GROUPS defined.")


In [ ]:
# ── R packages (run once in R console, then comment out) ─────────────────────
# BiocManager::install(c("sva", "limma", "batchelor", "RUVSeq", "qsmooth",
#                        "preprocessCore", "missForest", "softImpute", "oposSOM"))
# devtools::install_github("immunogenomics/harmony")
# devtools::install_github("jenniferfranks/FSQN")

# ── Python packages ───────────────────────────────────────────────────────────
# pip install harmonypy combat inmoose scanorama bbknn fancyimpute tqdm
print("Setup complete.")

## 0 Check whether imputation before log-transform had spoiled the data

In [ ]:

import gzip
import io
import os


import boto3

S3_BUCKET = "$FL_S3_BUCKET"
S3_PREFIX = "FL_batch_correction"

S3_EXP_KEY_KNN = f"{S3_PREFIX}/prepared/S0_no_removal__knn__exp.tsv.gz"
S3_EXP_KEY_STRICT = f"{S3_PREFIX}/prepared/S0_no_removal__strict__exp.tsv.gz"
S3_EXP_KEY_SOFTIMPUTE = f"{S3_PREFIX}/prepared/S0_no_removal__softimpute__exp.tsv.gz"

comb_ann = pd.read_csv(ANN_PATH, index_col=0)
comb_ann = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]

s3 = boto3.client("s3")

obj  = s3.get_object(Bucket=S3_BUCKET, Key=S3_EXP_KEY_STRICT)
body = obj["Body"].read()
body = gzip.decompress(body)
comb_exp_strict = pd.read_csv(io.BytesIO(body), sep="\t", index_col=0)

common_samples = comb_exp_strict.index.intersection(comb_ann.index) 
comb_exp_strict = comb_exp_strict.loc[common_samples]#.dropna(axis=1)
comb_exp_strict = comb_exp_strict.loc[~comb_exp_strict.index.duplicated(keep="first"), :]


obj  = s3.get_object(Bucket=S3_BUCKET, Key=S3_EXP_KEY_KNN)
body = obj["Body"].read()
body = gzip.decompress(body)
comb_exp_knn = pd.read_csv(io.BytesIO(body), sep="\t", index_col=0)
comb_exp_knn = comb_exp_knn.loc[common_samples]#.dropna(axis=1)
comb_exp_knn = comb_exp_knn.loc[~comb_exp_knn.index.duplicated(keep="first"), :]


obj  = s3.get_object(Bucket=S3_BUCKET, Key=S3_EXP_KEY_SOFTIMPUTE)
body = obj["Body"].read()
body = gzip.decompress(body)
comb_exp_softimpute = pd.read_csv(io.BytesIO(body), sep="\t", index_col=0)
comb_exp_softimpute = comb_exp_softimpute.loc[common_samples]#.dropna(axis=1)
comb_exp_softimpute = comb_exp_softimpute.loc[~comb_exp_softimpute.index.duplicated(keep="first"), :]




In [ ]:
comb_exp_strict

In [ ]:
comb_exp_knn.dropna(axis=1)

In [ ]:
comb_exp_softimpute.dropna(axis=1)

## 1 — Data Loading and Baseline Description


In [ ]:
# ── Load raw expressions and annotations ─────────────────────────────────────
comb_exp = pd.read_csv(EXP_PATH, sep="\t", index_col=0)
comb_ann = pd.read_csv(ANN_PATH, index_col=0)

comb_ann['Diagnosis_with_coo'] = comb_ann.Diagnosis_cell_type_general + '_' + comb_ann['coo_bg'].fillna('')
comb_ann['Diagnosis_unified_with_coo'] = comb_ann.Diagnosis_cell_type_unified + '_' + comb_ann['coo_bg'].fillna('')
comb_ann.loc[comb_ann.index.isin(['PUB_Suntsova_GSE120795']), "RNA_BATCH"] = "RNASeq_FF_Unknown"

#comb_ann[comb_ann.COHORT_LABEL == 'PUB_Suntsova_GSE120795']

common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
comb_exp = comb_exp.loc[common_samples]#.dropna(axis=1)
comb_exp = comb_exp.loc[~comb_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
comb_ann = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]



print(f"Raw: {comb_exp.shape[0]} samples × {comb_exp.shape[1]} genes")
print(f"\n{BATCH_COL} distribution:")
print(comb_ann[BATCH_COL].value_counts().to_string())
print(f"\n{BIO_COL} distribution:")
print(comb_ann[BIO_COL].value_counts().to_string())

In [ ]:
# ── Baseline batch effect quantification (before any filtering/normalization) ─
# log_transform_by_cohort, pca_variance_explained_by_batch, r2_batch
# imported from harmonization-scripts/bench_shared

baseline_r2 = r2_batch(comb_exp, comb_ann)
print(f"Baseline batch R\u00b2 in PCA (before normalization): {baseline_r2:.2%}")


In [ ]:
r2_batch(comb_exp, comb_ann, BIO_COL)

## 2 — Dataset Filtering Strategies

**10 strategies** covering the full range from zero removal to platform-level filtering.  
Hard constraint: ≤ 1/3 of total samples removed (≤ 2,413 from 7,238; ≥ 4,825 retained).  
Strategy C is included as an upper-bound reference and will typically exceed the budget.


In [ ]:
# ── 2a. Invariant biological exclusions (always applied) ──────────────────────
# RARE_GROUPS imported from harmonization-scripts/bench_shared

ann_base = comb_ann[~comb_ann[BIO_COL].isin(RARE_GROUPS)].copy()
print(f"After rare-group removal: {len(ann_base)} samples")

# ── Budget constants ──────────────────────────────────────────────────────────
TOTAL_SAMPLES = len(comb_ann)
MAX_REMOVE    = TOTAL_SAMPLES // 3
MIN_RETAINED  = TOTAL_SAMPLES - MAX_REMOVE
print(f"Budget: remove at most {MAX_REMOVE} ({MAX_REMOVE/TOTAL_SAMPLES:.1%}), retain ≥ {MIN_RETAINED}")

In [ ]:
# ── Filter helper and strategy constants ──────────────────────────────────────
# apply_filter, NORMAL_GROUPS imported from harmonization-scripts/bench_shared

RNASEQ_BATCHES     = [b for b in ann_base[BATCH_COL].unique() if b.startswith("RNASeq")]
MICROARRAY_BATCHES = [b for b in ann_base[BATCH_COL].unique() if not b.startswith("RNASeq")]
AFFYMETRIX_BATCHES = [b for b in ann_base[BATCH_COL].unique() if b.startswith("GPL570")]

# ── Strategy S0 — no removal (absolute baseline) ─────────────────────────────
ann_S0 = ann_base.copy()
print(f"Strategy S0 — no removal: {len(ann_S0)} samples")


In [ ]:
# ── Strategy A — confirmed bad batches (from prior analysis) ──────────────────
# BAD_BATCHES_A, BAD_COHORTS_A, BAD_BATCHES_B, BAD_COHORTS_B
# imported from harmonization-scripts/bench_shared
print("Strategy A — confirmed bad batches:")
ann_A = apply_filter(ann_base, bad_batches=BAD_BATCHES_A, bad_cohorts=BAD_COHORTS_A)

# ── Strategy B — extended bad batches (aggressive) ────────────────────────────
print("Strategy B — extended bad batches:")
ann_B = apply_filter(ann_base, bad_batches=BAD_BATCHES_B, bad_cohorts=BAD_COHORTS_B)


In [ ]:
# ── Strategy C — RNA-seq only (reference; may exceed budget) ──────────────────
print("Strategy C — RNA-seq only (reference, may exceed budget):")
ann_C = apply_filter(ann_base, keep_batches=RNASEQ_BATCHES)

# ── Strategy D — malignancies only (no normal B cells) ───────────────────────
MALIGNANT_GROUPS = [g for g in ann_base[BIO_COL].unique() if g not in NORMAL_GROUPS]

print("Strategy D — malignancies only (no normal B cells):")
ann_D = apply_filter(ann_base, keep_groups=MALIGNANT_GROUPS)

In [ ]:
# ── Strategy F — microarray only (all RNA-seq excluded) ──────────────────────
print("Strategy F — microarray only (RNA-seq excluded):")
ann_F = apply_filter(ann_base, keep_batches=MICROARRAY_BATCHES)

# ── Strategy G — Affymetrix GPL570 only ──────────────────────────────────────
print("Strategy G — Affymetrix GPL570 only:")
ann_G = apply_filter(ann_base, keep_batches=AFFYMETRIX_BATCHES)

In [ ]:
# ── Strategies E1–E3 — iterative PCA-based outlier removal ───────────────────
# identify_outlier_batches, prepare_dataset imported from harmonization-scripts/bench_shared


In [ ]:
# ── Iterative removal rounds (run interactively; inspect PCA after each round) ─

# Round E1: start from Strategy A, find and remove top-2 outlier batches
print("Iterative round E1:")
exp_A_prep, ann_A_prep = prepare_dataset(ann_A, comb_exp)
outliers_E1 = identify_outlier_batches(exp_A_prep, ann_A_prep)
ann_E1 = apply_filter(ann_A, bad_batches=BAD_BATCHES_A + outliers_E1,
                      bad_cohorts=BAD_COHORTS_A)

# Round E2: continue from E1
print("\nIterative round E2:")
exp_E1, ann_E1_prep = prepare_dataset(ann_E1, comb_exp)
outliers_E2 = identify_outlier_batches(exp_E1, ann_E1_prep)
ann_E2 = apply_filter(ann_E1, bad_batches=outliers_E2)

# Round E3: continue from E2
print("\nIterative round E3:")
exp_E2, ann_E2_prep = prepare_dataset(ann_E2, comb_exp)
outliers_E3 = identify_outlier_batches(exp_E2, ann_E2_prep)
ann_E3 = apply_filter(ann_E2, bad_batches=outliers_E3)

In [ ]:
# ── Pre-normalization batch R² after each iterative round ────────────────────
for label, ann_iter in [("E1", ann_E1), ("E2", ann_E2), ("E3", ann_E3)]:
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    r2 = r2_batch(exp_iter, ann_p)
    print(f"  Strategy {label}: {len(ann_iter)} samples, batch R² = {r2:.3f}")


In [ ]:
# ── PCA visualisation after each iterative round ─────────────────────────────
for label, ann_iter in [("S0_base", ann_base), ("A", ann_A),
                         ("E1", ann_E1), ("E2", ann_E2), ("E3", ann_E3)]:
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    pca_plot(exp_iter, grouping=ann_p[BATCH_COL].fillna("NA"),
             palette=rna_batch_palette, legend=None, ax=axs[0],
             title=f"{label} — {BATCH_COL}")
    pca_plot(exp_iter, grouping=ann_p[BIO_COL].fillna("NA"),
             palette=lymphoma_ontogeny_palette, legend=None, ax=axs[1],
             title=f"{label} — {BIO_COL}")
    plt.suptitle(f"Iterative removal — Strategy {label}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── tSNE visualisation after each iterative round ─────────────────────────────
for label, ann_iter in [("S0_base", ann_base), ("A", ann_A),
                         ("E1", ann_E1), ("E2", ann_E2), ("E3", ann_E3)]:
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    tsne_plot(exp_iter, grouping=ann_p[BATCH_COL].fillna("NA"),
             palette=rna_batch_palette, legend=None, ax=axs[0],
             title=f"{label} — {BATCH_COL}")
    tsne_plot(exp_iter, grouping=ann_p[BIO_COL].fillna("NA"),
             palette=lymphoma_ontogeny_palette, legend=None, ax=axs[1],
             title=f"{label} — {BIO_COL}")
    plt.suptitle(f"Iterative removal — Strategy {label}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Strategy registry ─────────────────────────────────────────────────────────
FILTER_STRATEGIES = {
    "S0_no_removal":     ann_S0,
    "A_confirmed_bad":   ann_A,
    "B_extended_bad":    ann_B,
    "C_rnaseq_only":     ann_C,   # reference only, may exceed budget
    "D_malignant_only":  ann_D,
    "E1_iterative_r1":   ann_E1,
    "E2_iterative_r2":   ann_E2,
    "E3_iterative_r3":   ann_E3,
    "F_microarray_only": ann_F,
    "G_affymetrix_only": ann_G,
}

# ── Prepare expression matrix for all strategies ──────────────────────────────
datasets = {}
for name, ann_strat in FILTER_STRATEGIES.items():
    print(f"\n── {name} ──")
    exp_s, ann_s = prepare_dataset(ann_strat, comb_exp)
    datasets[name] = (exp_s, ann_s)

### PCA & tSNE for all sample selection strategies

In [ ]:
# ── PCA visualisation after each iterative round ─────────────────────────────
for label, ann_iter in FILTER_STRATEGIES.items():
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    exp_iter = log_transform_by_cohort(exp_iter, ann_p, batch_col='COHORT_LABEL', threshold=30)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    pca_plot(exp_iter, grouping=ann_p[BATCH_COL].fillna("NA"),
             palette=rna_batch_palette, legend=None, ax=axs[0],
             title=f"{label} — {BATCH_COL}")
    pca_plot(exp_iter, grouping=ann_p[BIO_COL].fillna("NA"),
             palette=lymphoma_ontogeny_palette, legend=None, ax=axs[1],
             title=f"{label} — {BIO_COL}")
    plt.suptitle(f"Iterative removal — Strategy {label}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── tSNE visualisation after each iterative round ─────────────────────────────
for label, ann_iter in FILTER_STRATEGIES.items():
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    exp_iter = log_transform_by_cohort(exp_iter, ann_p, batch_col='COHORT_LABEL', threshold=30)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    tsne_plot(exp_iter, grouping=ann_p[BATCH_COL].fillna("NA"),
             palette=rna_batch_palette, legend=None, ax=axs[0],
             title=f"{label} — {BATCH_COL}")
    tsne_plot(exp_iter, grouping=ann_p[BIO_COL].fillna("NA"),
             palette=lymphoma_ontogeny_palette, legend=None, ax=axs[1],
             title=f"{label} — {BIO_COL}")
    plt.suptitle(f"Iterative removal — Strategy {label}", fontsize=10)
    plt.tight_layout()
    plt.show()

### Distribution plots

In [ ]:
# ── 2c. Per-batch expression distribution check (Strategy A) ─────────────────
def plot_batch_distributions(exp_df, ann_df, batch_col=BATCH_COL,
                              ncols=8, figsize=(28, 20), title_prefix=""):
    batches = ann_df[batch_col].value_counts().index
    nrows   = int(np.ceil(len(batches) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for ax, batch in zip(axs.flatten(), batches):
        samples = ann_df.index[ann_df[batch_col] == batch]
        vals    = exp_df.reindex(samples).values#.ravel()
        vals    = vals[~np.isnan(vals)]
        ax.hist(vals, bins=40, density=True, alpha=0.7, color="steelblue")
        ax.set_title(batch, fontsize=10)
        #ax.set_xlim(-2, 18)
    for ax in axs.flatten()[len(batches):]:
        ax.set_visible(False)
    fig.suptitle(f"{title_prefix} Expression distributions per {BATCH_COL}", fontsize=10)
    plt.tight_layout()
    return fig

#exp_A, ann_A_p = datasets["A_confirmed_bad"]
exp_A, ann_A_p = comb_exp_strict, comb_ann
exp_A = log_transform_by_cohort(exp_A, ann_A_p)
plot_batch_distributions(exp_A, ann_A_p, batch_col='COHORT_LABEL', title_prefix="Strategy A —",
                        figsize=(28, 20))
plt.show()

In [ ]:
#KNN

exp_A, ann_A_p = comb_exp_knn, comb_ann
exp_A = log_transform_by_cohort(exp_A, ann_A_p)
plot_batch_distributions(exp_A, ann_A_p, batch_col='COHORT_LABEL', title_prefix="Strategy A —",
                        figsize=(28, 20))
plt.show()

In [ ]:
#SOFTIMPUTE
exp_A, ann_A_p = comb_exp_softimpute, comb_ann
exp_A = log_transform_by_cohort(exp_A, ann_A_p)
plot_batch_distributions(exp_A, ann_A_p, batch_col='COHORT_LABEL', title_prefix="Strategy A —",
                        figsize=(28, 20))
plt.show()

In [ ]:
for label, ann_iter in FILTER_STRATEGIES.items():
    exp_iter, ann_p = prepare_dataset(ann_iter, comb_exp)
    exp_iter = log_transform_by_cohort(exp_iter, ann_p)
    plot_batch_distributions(exp_iter, ann_p, title_prefix=label,  batch_col='COHORT_LABEL', figsize=(14, len(ann_p['COHORT_LABEL'].unique())/8))
    plt.show()

### NA Imputation approaches

In [ ]:
# ── 2e. Gene filtering — imputation alternatives ──────────────────────────────
# prepare_dataset_imputed imported from harmonization-scripts/bench_shared


In [ ]:
# knn lasts ~5 min, missforest > 25 min. Skip it

## 3 — Harmonization Methods

25 methods ordered **low → medium → high harshness**.

| Tier | Methods |
|---|---|
| Low | raw, median scaling, limma, SVA, TMM\*, VST\* |
| Medium | ComBat, ComBat-seq, pyComBat, InMoose/pycombat_seq, RUVSeq, MNN, Harmony, Scanorama, FSMVN, HarmonizR, PEER† |
| High | qsmooth, FSQN-py, FSQN-R ★, quantile, rank, TDM, Shambhala2, Angel |

\* RNA-seq only (raises NotImplementedError on mixed-platform data).  
† PEER raises NotImplementedError (unavailable for R 4.5).

All functions share the signature `fn(exp_df, ann_df, **kw) → pd.DataFrame`.  
`bio_col=BIO_COL` and `batch_col=BATCH_COL` are defaults throughout.


In [ ]:
# ── Normalization functions — loaded from harmonization-scripts/bench_shared ──
# All normalize_* functions are defined in bench_shared.py and imported above.
# Low:    normalize_raw, normalize_median_scaling, normalize_limma, normalize_sva


In [ ]:
# Medium: normalize_combat, normalize_combat_seq, normalize_pycombat,
#         normalize_inmoose_combat_seq, normalize_ruv, normalize_mnn,
#         normalize_harmony, normalize_scanorama, normalize_fsmvn


In [ ]:
# High:   normalize_qsmooth, normalize_fsqn_py, normalize_fsqn_r,
#         normalize_quantile, normalize_rank, normalize_tdm, normalize_shambhala
# Extra:  normalize_harmonizr, normalize_tmm, normalize_vst, normalize_peer


In [ ]:
# ── Method registry — imported from harmonization-scripts/bench_shared ────────
# METHODS and HARSHNESS_COLORS were imported in the setup cell above.
print(f"{len(METHODS)} methods registered.")


In [ ]:
# ── 3.20 Smoke test: verify all methods on 100 samples × 500 genes ──────────
import traceback as _tb
print("Smoke-testing all methods on 100 samples × 500 genes subset...")
_ann_smoke = comb_ann.dropna(subset=[BATCH_COL, BIO_COL]).head(100)
_common_g  = comb_exp.dropna(axis=1).columns[:500]
_exp_smoke = comb_exp.loc[_ann_smoke.index, _common_g]

_smoke_ok, _smoke_fail = [], []
for _name, (_fn, _harshness) in METHODS.items():
    try:
        _fn(_exp_smoke, _ann_smoke, batch_col=BATCH_COL, bio_col=BIO_COL)
        print(f"  ✓ {_name}")
        _smoke_ok.append(_name)
    except NotImplementedError as _e:
        print(f"  ✓ {_name} (RNA-seq-only gate, expected on mixed data)")
        _smoke_ok.append(_name)
    except Exception:
        print(f"  ✗ {_name}:")
        print(_tb.format_exc())
        _smoke_fail.append(_name)

print(f"\n{len(_smoke_ok)}/{len(METHODS)} methods passed smoke test.")
if _smoke_fail:
    print(f"Failed: {_smoke_fail}")


## 4 — Full Cross-Product Evaluation (All Strategies × Imputation × Methods × Post-Removal)

**4-dimensional cross-product** covering all combinations:

| Dimension | Options | Count |
|---|---|---|
| **D1** — Sample-removal strategy | S0, A, B, C, D, E1, E2, E3, F, G | 10 |
| **D2** — Imputation approach | strict (dropna), KNN, missForest, softImpute | 4 |
| **D3** — Harmonization method | 25 methods (low → medium → high harshness) | 25 |
| **D4** — Post-harmonization outlier removal | skip / apply `identify_outlier_batches` once | 2 |

Total: **10 × 4 × 25 × 2 = 2,000 combinations**.  
Key: `cross_results[(strategy, imputation, method, post_removal)]`.  
Top `TOP_K_SOM` combinations carry forward to SOM.


In [ ]:
# ── 4a. Pre-compute imputed datasets for all (strategy, imputation) pairs ─────
IMPUTATION_METHODS  = ["strict", "knn", "missforest", "softimpute"]
POST_REMOVAL_OPTIONS = [False, True]

imputed_datasets = {}  # (strat_name, imp) -> (exp_df, ann_df)
for _strat_name, _ann_strat in tqdm(FILTER_STRATEGIES.items(), desc="Imputing"):
    imputed_datasets[(_strat_name, "strict")] = datasets[_strat_name]
    for _imp in ["knn", "missforest", "softimpute"]:
        try:
            imputed_datasets[(_strat_name, _imp)] = prepare_dataset_imputed(
                _ann_strat, comb_exp, method=_imp
            )
        except Exception as _e:
            print(f"  Imputation FAILED {_strat_name} × {_imp}: {_e}")
            imputed_datasets[(_strat_name, _imp)] = datasets[_strat_name]

# ── 4a. Run full 4D cross-product ─────────────────────────────────────────────
# cross_results[(strategy, imputation, method, post_removal)] =
#     {"exp", "ann", "r2_batch", "r2_diag", "harshness"}
cross_results = {}

for _strat_name in tqdm(FILTER_STRATEGIES, desc="Strategies"):
    for _imp in IMPUTATION_METHODS:
        _exp_s, _ann_s = imputed_datasets[(_strat_name, _imp)]
        for _method_name, (_fn, _harshness) in METHODS.items():
            _exp_norm = None
            try:
                _exp_norm = _fn(_exp_s, _ann_s, batch_col=BATCH_COL, bio_col=BIO_COL)
            except Exception as _e:
                print(f"  FAILED {_strat_name}×{_imp}×{_method_name}: {_e}")
                import traceback; print(traceback.format_exc())
                for _pr in POST_REMOVAL_OPTIONS:
                    cross_results[(_strat_name, _imp, _method_name, _pr)] = {
                        "exp": None, "ann": _ann_s,
                        "r2_batch": np.nan, "r2_diag": np.nan,
                        "harshness": _harshness,
                    }
                continue
            for _post_rm in POST_REMOVAL_OPTIONS:
                try:
                    _exp_out = _exp_norm.copy()
                    _ann_out = _ann_s
                    if _post_rm:
                        _outliers = identify_outlier_batches(_exp_out, _ann_out)
                        if _outliers:
                            _ann_out = _ann_out[~_ann_out[BATCH_COL].isin(_outliers)]
                            _exp_out = _exp_out.loc[_ann_out.index]
                    _r2_b = r2_batch(_exp_out, _ann_out, batch_col=BATCH_COL)
                    _r2_d = r2_batch(_exp_out, _ann_out, batch_col=BIO_COL)
                    cross_results[(_strat_name, _imp, _method_name, _post_rm)] = {
                        "exp": _exp_out, "ann": _ann_out,
                        "r2_batch": _r2_b, "r2_diag": _r2_d,
                        "harshness": _harshness,
                    }
                except Exception as _e:
                    print(f"  FAILED post-rm {_strat_name}×{_imp}×{_method_name}×{_post_rm}: {_e}")
                    cross_results[(_strat_name, _imp, _method_name, _post_rm)] = {
                        "exp": None, "ann": _ann_s,
                        "r2_batch": np.nan, "r2_diag": np.nan,
                        "harshness": _harshness,
                    }

print(f"Cross-product complete: {len(cross_results)} combinations evaluated.")


In [ ]:
# ── 4b. Cross-product batch R² heatmap (collapsed: mean over D2 and D4) ────────
batch_r2_matrix = pd.DataFrame(
    index=list(FILTER_STRATEGIES.keys()), columns=list(METHODS.keys()), dtype=float
)
diag_r2_matrix  = pd.DataFrame(
    index=list(FILTER_STRATEGIES.keys()), columns=list(METHODS.keys()), dtype=float
)

for _strat in FILTER_STRATEGIES:
    for _meth in METHODS:
        _vals_b = [cross_results[(_strat, _imp, _meth, _pr)]["r2_batch"]
                   for _imp in IMPUTATION_METHODS
                   for _pr in POST_REMOVAL_OPTIONS
                   if (_strat, _imp, _meth, _pr) in cross_results]
        _vals_d = [cross_results[(_strat, _imp, _meth, _pr)]["r2_diag"]
                   for _imp in IMPUTATION_METHODS
                   for _pr in POST_REMOVAL_OPTIONS
                   if (_strat, _imp, _meth, _pr) in cross_results]
        batch_r2_matrix.loc[_strat, _meth] = np.nanmean(_vals_b) if _vals_b else np.nan
        diag_r2_matrix.loc[_strat, _meth]  = np.nanmean(_vals_d) if _vals_d else np.nan

fig, axs = plt.subplots(1, 2, figsize=(22, 6))
sns.heatmap(batch_r2_matrix.astype(float), ax=axs[0], cmap="RdYlGn_r",
            vmin=0, vmax=1, annot=True, fmt=".2f", linewidths=0.3,
            cbar_kws={"label": "Mean Batch R² across D2×D4 (↓ better)"})
axs[0].set_title("Pre-SOM Batch R² — Strategy × Method (mean over imputation × post-removal)")
axs[0].set_xlabel("Normalization method"); axs[0].set_ylabel("Filtering strategy")

sns.heatmap(diag_r2_matrix.astype(float), ax=axs[1], cmap="RdYlGn",
            vmin=0, vmax=0.5, annot=True, fmt=".2f", linewidths=0.3,
            cbar_kws={"label": f"{BIO_COL} R² (↑ better)"})
axs[1].set_title(f"Pre-SOM Diagnosis R² — Strategy × Method")
axs[1].set_xlabel("Normalization method"); axs[1].set_ylabel("Filtering strategy")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crossproduct_heatmap.pdf", bbox_inches="tight")
plt.show()

batch_r2_matrix.to_csv(f"{OUT_DIR}/crossproduct_batch_r2.csv")
diag_r2_matrix.to_csv(f"{OUT_DIR}/crossproduct_diag_r2.csv")

# Flat ranked table of all 2,000 combinations
_flat_rows = []
for (_s, _i, _m, _p), _v in cross_results.items():
    if not np.isnan(_v["r2_batch"]):
        _flat_rows.append({"strategy": _s, "imputation": _i, "method": _m,
                           "post_removal": _p, "r2_batch": _v["r2_batch"],
                           "r2_diag": _v["r2_diag"]})
flat_df = pd.DataFrame(_flat_rows).sort_values("r2_batch")
flat_df.to_csv(f"{OUT_DIR}/crossproduct_all_combinations.csv", index=False)
print(f"Best 10 combinations (all 4D):\n{flat_df.head(10).to_string(index=False)}")


In [ ]:
# ── 4b-note. Angel gene-space caveat ─────────────────────────────────────────
_angel_n_genes = [
    cross_results[(s, i, "25_angel", pr)].get("n_genes", float("nan"))
    for s in FILTER_STRATEGIES
    for i in IMPUTATION_METHODS
    for pr in POST_REMOVAL_OPTIONS
    if (s, i, "25_angel", pr) in cross_results
    and cross_results[(s, i, "25_angel", pr)].get("status") == "ok"
]
_other_n_genes = [
    cross_results[(s, i, m, pr)].get("n_genes", float("nan"))
    for s in FILTER_STRATEGIES
    for i in IMPUTATION_METHODS
    for m in METHODS if m != "25_angel"
    for pr in POST_REMOVAL_OPTIONS
    if (s, i, m, pr) in cross_results
    and cross_results[(s, i, m, pr)].get("status") == "ok"
]
if _angel_n_genes:
    _med_angel = float(np.nanmedian(_angel_n_genes))
    _med_other = float(np.nanmedian(_other_n_genes))
    print(
        f"Note \u2014 25_angel retains {_med_angel:.0f} / {_med_other:.0f} genes on "
        f"average ({100*_med_angel/_med_other:.1f}% of the full gene set). "
        f"Its PCA R\u00b2 is computed on this reduced, platform-stable gene space "
        f"and is not directly comparable with other methods in the heatmap above."
    )


In [ ]:
# ── 4c. Summary bar chart per method (mean across all strategies/D2/D4) ────────
mean_batch_r2 = batch_r2_matrix.mean(axis=0).sort_values()
colors_bar = [HARSHNESS_COLORS[METHODS[m][1]] for m in mean_batch_r2.index]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(mean_batch_r2)), mean_batch_r2.values, color=colors_bar, alpha=0.85)
ax.set_xticks(range(len(mean_batch_r2)))
ax.set_xticklabels(mean_batch_r2.index, rotation=40, ha="right", fontsize=8)
ax.axhline(0.16, ls="--", lw=1, color="gray", label="FSQN R baseline (16%)")
legend_patches = [Patch(color=c, label=l) for l, c in HARSHNESS_COLORS.items()]
ax.legend(handles=legend_patches)
ax.set_ylabel("Mean batch R² across all strategies / imputation / post-removal")
ax.set_title("Pre-SOM Method Ranking — Mean Across All Combinations")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/method_ranking_bar.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ── 4d. Select top combinations for SOM (4-tuple key) ────────────────────────
combo_rows = []
for (_strat, _imp, _meth, _post_rm), _v in cross_results.items():
    if not np.isnan(_v["r2_batch"]):
        combo_rows.append({
            "strategy":    _strat,
            "imputation":  _imp,
            "method":      _meth,
            "post_removal": _post_rm,
            "r2_batch":    _v["r2_batch"],
            "r2_diag":     _v["r2_diag"],
            "harshness":   METHODS[_meth][1],
            "n_samples":   len(_v["ann"]),
        })

combo_df   = pd.DataFrame(combo_rows).sort_values("r2_batch").reset_index(drop=True)
top_combos = combo_df.head(TOP_K_SOM)
print(f"Top {TOP_K_SOM} combinations selected for SOM:")
print(top_combos[["strategy", "imputation", "method", "post_removal",
                   "r2_batch", "harshness", "n_samples"]].to_string())


In [ ]:
# ── 4e. PCA / UMAP / tSNE for top combinations ───────────────────────────────
PRIMARY_COLS = [
    (BATCH_COL, rna_batch_palette),
    (BIO_COL,   lymphoma_ontogeny_palette),
]
SECONDARY_COLS = [
    ("COHORT_LABEL",  cohort_palette),
    ("PLATFORM_RNA",  platform_palette),
    ("RNASEQ_SOURCE", None),
    ("TUMOR_NORMAL",  None),
    ("Major_group",   lymphoma_ontogeny_palette),
]
COLOR_COLS = PRIMARY_COLS + SECONDARY_COLS

for _col, _pal in COLOR_COLS:
    _combos_show = top_combos if (_col, _pal) in PRIMARY_COLS else top_combos.head(8)
    _n = len(_combos_show)
    _fig, _axs = plt.subplots(_n, 3, figsize=(18, 3.5 * _n), squeeze=False)
    for _ri, (_, _row) in enumerate(_combos_show.iterrows()):
        _strat, _imp = _row["strategy"], _row["imputation"]
        _meth, _pr   = _row["method"],   _row["post_removal"]
        _key   = (_strat, _imp, _meth, _pr)
        _label = f"{_strat}×{_imp}×{_meth}×{_pr}"
        _exp_norm = cross_results[_key]["exp"]
        _ann_s    = cross_results[_key]["ann"]
        if _exp_norm is None:
            continue
        _grouping = _ann_s.loc[_exp_norm.index, _col].fillna("NA")
        _pg = PALETTE_GROUPS.get(f"{_pal}_palette") if _pal else None
        pca_plot(_exp_norm,  grouping=_grouping, palette=_pal, legend=None,
                 ax=_axs[_ri, 0], title=f"{_label} — PCA", palette_groups=_pg)
        umap_plot(_exp_norm, grouping=_grouping, palette=_pal, legend=None,
                  ax=_axs[_ri, 1], title=f"{_label} — UMAP", palette_groups=_pg)
        tsne_plot(_exp_norm, grouping=_grouping, palette=_pal, legend=None,
                  ax=_axs[_ri, 2], title=f"{_label} — tSNE", palette_groups=_pg)
    _fig.suptitle(f"Pre-SOM Top Combinations — Color: {_col}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/pre_som_top_{_col}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# ── 4f. Expression distribution comparison for top combinations ──────────────
def distribution_panel_comparison(combo_list, cross_res, batch_col=BATCH_COL):
    """Per-batch histograms for top 4D combinations."""
    batches   = comb_ann[batch_col].value_counts().head(20).index
    n_combos  = len(combo_list)
    n_batches = len(batches)
    fig, axs  = plt.subplots(n_combos, n_batches,
                              figsize=(n_batches * 1.5, n_combos * 1.2),
                              squeeze=False)
    for mi, (_, row) in enumerate(combo_list.iterrows()):
        strat, imp  = row["strategy"], row["imputation"]
        meth, pr    = row["method"],   row["post_removal"]
        key         = (strat, imp, meth, pr)
        v           = cross_res[key]
        exp_norm    = v["exp"]
        ann_s       = v["ann"]
        harshness   = METHODS[meth][1]
        if exp_norm is None:
            continue
        for bi, batch in enumerate(batches):
            ax   = axs[mi, bi]
            samp = ann_s.index[ann_s[batch_col] == batch]
            vals = exp_norm.reindex(samp).values.ravel()
            vals = vals[~np.isnan(vals)]
            ax.hist(vals, bins=30, density=True, alpha=0.75,
                    color=HARSHNESS_COLORS[harshness])
            if mi == 0: ax.set_title(batch[:16], fontsize=5, rotation=30)
            if bi == 0: ax.set_ylabel(f"{strat[:10]}\n{meth[:10]}", fontsize=5)
            ax.set_xlim(-2, 18); ax.tick_params(labelsize=4)
    plt.suptitle("Expression distributions per batch × top combinations", fontsize=9)
    plt.tight_layout()
    return fig

distribution_panel_comparison(top_combos, cross_results)
plt.savefig(f"{OUT_DIR}/pre_som_distributions_top.pdf", bbox_inches="tight")
plt.show()


## 4g — Optional: Angel Platform-Gene Filter

Applies the gene-selection step from Angel (PLOS Comput Biol 2020) as an optional
post-normalization filter to any harmonized expression matrix.  Set `APPLY_ANGEL_FILTER = True`
to remove platform-biased genes before Section 5 SOM analysis.

This is independent of the `25_angel` benchmark entry — it can be applied on top of *any*
normalization (e.g., FSQN, ComBat, quantile).  When `APPLY_ANGEL_FILTER = False`, downstream
cells receive `exp_filtered` as an unmodified alias for `exp`.

In [ ]:
# ── 4g. Angel platform-gene filter (optional) ───────────────────────────────
APPLY_ANGEL_FILTER = False   # set True to remove platform-biased genes
ANGEL_THRESHOLD    = 0.20    # per-gene platform R² threshold; paper default

In [ ]:
from bench_shared import _angel_platform_variance

# Set these to the (strat, imp, method, post_rm) combo you want to apply the filter to.
# Example: the top-ranked combo from Cell 4d (combo_df.iloc[0]).
_ANGEL_STRAT   = "A_confirmed_bad"
_ANGEL_IMP     = "strict"
_ANGEL_METHOD  = "16_fsqn_r"
_ANGEL_POST_RM = False

if APPLY_ANGEL_FILTER:
    from load_cross_product_results import load_exp
    exp_best = load_exp(_ANGEL_STRAT, _ANGEL_IMP, _ANGEL_METHOD, post_rm=_ANGEL_POST_RM)
    ann_best = comb_ann.loc[exp_best.index]

    _batch_series = ann_best.loc[exp_best.index, "RNA_BATCH"].astype(str)
    _gene_r2      = _angel_platform_variance(exp_best, _batch_series)
    _stable       = _gene_r2[_gene_r2 < ANGEL_THRESHOLD].index

    n_kept    = len(_stable)
    n_total   = len(_gene_r2)
    n_dropped = n_total - n_kept
    print(
        f"Angel filter (threshold={ANGEL_THRESHOLD}): "
        f"kept {n_kept} / {n_total} genes ({100*n_kept/n_total:.1f}%); "
        f"removed {n_dropped} platform-biased genes."
    )

    removed_genes = _gene_r2[~_gene_r2.index.isin(_stable)].sort_values(ascending=False)
    print("Top 20 most platform-biased genes (removed):")
    display(removed_genes.head(20).rename("platform_R2").to_frame())

    exp_filtered = exp_best[_stable]

    from bench_shared import r2_batch
    r2_before = r2_batch(exp_best,      ann_best, batch_col="RNA_BATCH")
    r2_after  = r2_batch(exp_filtered,  ann_best, batch_col="RNA_BATCH")
    print(f"
Batch R² before Angel filter : {r2_before:.4f}")
    print(f"Batch R² after Angel filter  : {r2_after:.4f}")
    print(f"Δ = {r2_after - r2_before:+.4f}")
    print(
        "Note: post-filter R² reflects a reduced gene space — improvement is partly "
        "due to removing high-platform-variance genes, not correcting their values."
    )
else:
    exp_filtered = None   # not defined when filter is off; set manually if needed
    print("Angel filter not applied (APPLY_ANGEL_FILTER = False).")


## 5 — SOM Analysis (Top Combinations)

oposSOM is run only for the **top `TOP_K_SOM` combinations** ranked by pre-SOM batch R².  
Each SOM run takes ~40 min; results are cached to `harmonization_results/{run_name}/metagenes.csv`.


In [ ]:
# ── 5a. SOM runner: optional limma pre-correction + oposSOM ──────────────────
def run_som(exp_norm, ann_df, run_name,
            group_col=BIO_COL, apply_limma=True):
    """
    Full SOM pipeline: optional limma batch correction → oposSOM → metagene extraction.
    All oposSOM outputs (PDFs, HTML reports) are written to OUT_DIR/{run_name}/.
    Returns metagenes_df (metagenes × samples).
    """
    opossom_r = importr("oposSOM")
    limma_r   = importr("limma")
    base_r    = importr("base")
    common = exp_norm.index.intersection(ann_df.index)
    X      = exp_norm.loc[common].dropna(axis=1)
    ann    = ann_df.loc[X.index]
    if apply_limma:
        r_mat   = base_r.as_matrix(pandas2ri.py2rpy(X.T))
        batches = ro.StrVector(ann[BATCH_COL].astype(str).values)
        X_r     = limma_r.removeBatchEffect(r_mat, batch=batches)
        X_np    = X_r if isinstance(X_r, np.ndarray) else pandas2ri.rpy2py(X_r)
        X       = pd.DataFrame(X_np.T, index=X.index, columns=X.columns)
    r_X    = base_r.as_matrix(pandas2ri.py2rpy(X.T))
    groups = ann[group_col].astype(str).values
    out_dir  = f"{OUT_DIR}/{run_name}"
    orig_dir = os.getcwd()
    os.makedirs(out_dir, exist_ok=True)
    os.chdir(out_dir)
    pref = ro.ListVector({
        "dataset.name":                  "MyCohort+",
        "dim.1stLvlSom":                 "automatic",
        "standard.spot.modules":         "kmeans",
        "adjust.expression.values":      False,
        "feature.centralization":        True,
        "sample.quantile.normalization": True,
    })
    env = opossom_r.opossom_new(pref)
    env["indata"]       = r_X
    env["group.labels"] = ro.StrVector(groups)
    opossom_r.opossom_run(env)
    os.chdir(orig_dir)
    meta_r  = env["metadata"]
    meta_np = meta_r if isinstance(meta_r, np.ndarray) else pandas2ri.rpy2py(meta_r)
    meta_df = pd.DataFrame(meta_np, columns=X.index)
    meta_df.index = [f"M{i+1}" for i in range(meta_df.shape[0])]
    return meta_df, ann

In [ ]:
# ── 5b. Run SOM for top combinations (with caching) ──────────────────────────
# Runs are cached — re-running after interruption resumes from checkpoint.
# Expected time: ~40 min/run × TOP_K_SOM. Run overnight.

som_results = {}  # (strategy, imputation, method, post_removal) -> (metagenes_df, ann_df)

for _, combo_row in tqdm(top_combos.iterrows(), total=len(top_combos), desc="SOM runs"):
    strat, imp  = combo_row["strategy"], combo_row["imputation"]
    meth, pr    = combo_row["method"],   combo_row["post_removal"]
    key         = (strat, imp, meth, pr)
    run_name    = f"{strat}__{imp}__{meth}__{pr}"
    cache_path  = f"{OUT_DIR}/{run_name}/metagenes.csv"

    if os.path.exists(cache_path):
        meta_df = pd.read_csv(cache_path, index_col=0)
        som_results[key] = (meta_df, cross_results[key]["ann"])
        print(f"{run_name}: loaded from cache — {meta_df.shape[0]} metagenes")
        continue

    try:
        exp_norm = cross_results[key]["exp"]
        ann_s    = cross_results[key]["ann"]
        if exp_norm is None:
            print(f"{run_name}: SKIPPED (no expression data)")
            continue
        meta_df, ann_out = run_som(exp_norm, ann_s, run_name)
        meta_df.to_csv(cache_path)
        som_results[key] = (meta_df, ann_out)
        print(f"{run_name}: SOM complete — {meta_df.shape[0]} metagenes")
    except Exception as e:
        print(f"{run_name}: FAILED — {e}")


## 6 — Post-SOM Batch Effect Evaluation


In [ ]:
# ── 6a. Batch R² in metagene space ───────────────────────────────────────────
som_summary = {}
for (strat, imp, meth, pr), (meta_df, ann_out) in som_results.items():
    meta_T = meta_df.T
    r2_b   = r2_batch(meta_T, ann_out, batch_col=BATCH_COL)
    r2_d   = r2_batch(meta_T, ann_out, batch_col=BIO_COL)
    som_summary[(strat, imp, meth, pr)] = {
        "strategy": strat, "imputation": imp, "method": meth,
        "post_removal": pr,
        "post_batch_r2": r2_b, "post_diag_r2": r2_d,
    }

som_summary_df = pd.DataFrame(som_summary.values()).set_index(
    ["strategy", "imputation", "method", "post_removal"])
pre_metrics    = combo_df.set_index(
    ["strategy", "imputation", "method", "post_removal"])[["r2_batch", "r2_diag"]]
combined       = pre_metrics.join(som_summary_df, how="right").sort_values("post_batch_r2")
combined.to_csv(f"{OUT_DIR}/batch_effect_summary.csv")
print(combined.to_string())


In [ ]:
# ── 6b. PCA / UMAP / tSNE in metagene space ─────────────────────────────────
for _col, _pal in COLOR_COLS:
    _fig, _axs = plt.subplots(len(som_results), 3,
                               figsize=(18, 3.5 * len(som_results)),
                               squeeze=False)
    for _ri, ((strat, imp, meth, pr), (meta_df, ann_out)) in enumerate(som_results.items()):
        meta_T    = meta_df.T
        _grouping = ann_out.loc[meta_T.index, _col].fillna("NA")
        _label    = f"{strat}×{imp}×{meth}×{pr}"
        _pg = PALETTE_GROUPS.get(f"{_pal}_palette") if _pal else None
        pca_plot(meta_T,  grouping=_grouping, palette=_pal, legend=None,
                 ax=_axs[_ri, 0], title=f"{_label} — PCA", palette_groups=_pg)
        umap_plot(meta_T, grouping=_grouping, palette=_pal, legend=None,
                  ax=_axs[_ri, 1], title=f"{_label} — UMAP", palette_groups=_pg)
        tsne_plot(meta_T, grouping=_grouping, palette=_pal, legend=None,
                  ax=_axs[_ri, 2], title=f"{_label} — tSNE", palette_groups=_pg)
    _fig.suptitle(f"Post-SOM — Color: {_col}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/post_som_{_col}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# ── 6c. Formal batch-dominates-biology test ──────────────────────────────────
def batch_dominates_biology_test(meta_df, ann_df,
                                  batch_col=BATCH_COL,
                                  bio_col=BIO_COL,
                                  n_neighbors=15):
    """
    For each sample, find its k nearest neighbours in metagene space.
    PASSED if same_bio_ratio >= same_batch_ratio.
    """
    meta_T  = meta_df.T
    common  = meta_T.index.intersection(ann_df.index)
    X       = meta_T.loc[common].values
    batches = ann_df.loc[common, batch_col].fillna("NA").values
    bios    = ann_df.loc[common, bio_col].fillna("NA").values
    nn      = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
    _, idxs = nn.kneighbors(X)
    idxs    = idxs[:, 1:]
    same_batch = np.mean([np.mean(batches[i] == batches[idxs[i]]) for i in range(len(X))])
    same_bio   = np.mean([np.mean(bios[i]    == bios[idxs[i]])    for i in range(len(X))])
    passed = same_bio >= same_batch
    print(f"  same-batch NN: {same_batch:.3f}  |  same-bio NN: {same_bio:.3f}  "
          f"|  {'PASSED ✓' if passed else 'FAILED ✗'}")
    return {"same_batch": same_batch, "same_bio": same_bio, "passed": passed}

test_results = {}
for (strat, imp, meth, pr), (meta_df, ann_out) in som_results.items():
    print(f"\n── {strat}×{imp}×{meth}×{pr} ──")
    test_results[(strat, imp, meth, pr)] = batch_dominates_biology_test(meta_df, ann_out)


## 7 — SOM Portrait Comparison

The most important visual output. Portraits are compared:  
- **Across methods** (columns) for the same biological group (rows) — columns should agree on biology  
- **Across batches** within the same biology group — a successful method shows identical portraits per batch


In [ ]:
# ── 7a. Portrait helper functions ─────────────────────────────────────────────
def create_som_colormap():
    colors = ["darkblue", "blue", "cyan", "green", "yellow", "red", "darkred"]
    return LinearSegmentedColormap.from_list("som_rainbow", colors)


def group_portrait(meta_df, ann_df, group_label, group_col,
                   vmin=None, vmax=None, cmap=None, ax=None, title=None):
    """
    Draws the mean SOM portrait for all samples in a given group.
    meta_df: metagenes × samples
    """
    cmap    = cmap or create_som_colormap()
    samples = [s for s in ann_df.index[ann_df[group_col] == group_label]
               if s in meta_df.columns]
    if not samples:
        if ax: ax.set_visible(False)
        return
    mean_vec = meta_df[samples].mean(axis=1).values
    grid_sz  = int(np.ceil(np.sqrt(len(mean_vec))))
    portrait = mean_vec[:grid_sz**2].reshape(grid_sz, grid_sz, order="F")
    ax = ax or plt.gca()
    ax.imshow(portrait, cmap=cmap, origin="lower",
              vmin=vmin or np.percentile(mean_vec, 2),
              vmax=vmax or np.percentile(mean_vec, 98),
              aspect="auto")
    ax.set_title(title or group_label, fontsize=7)
    ax.axis("off")

In [ ]:
# ── 7b. Master portrait comparison grid ─────────────────────────────────────
PORTRAIT_GROUPS = [
    ("Diffuse_Large_B_Cell_Lymphoma",     BIO_COL),
    ("Follicular_Lymphoma",               BIO_COL),
    ("Diffuse_Large_B_Cell_Lymphoma_GCB", "Diagnosis_unified_with_coo"),
    ("Diffuse_Large_B_Cell_Lymphoma_ABC", "Diagnosis_unified_with_coo"),
    ("Normal_B_cells",                           "Major_group"),
    ("Kassandra",                         "Major_group"),
    ("RNASeq_FF_PolyA",                   BATCH_COL),
    ("GPL570_FF_Unknown",                 BATCH_COL),
    ("RNASeq_FFPE_Exome_capture",         BATCH_COL),
    ("GPL570_Unknown_Unknown",            BATCH_COL),
]

def plot_master_portrait_grid(som_res, groups, run_keys,
                               figsize_per_cell=(1.8, 1.8)):
    n_rows = len(groups)
    n_cols = len(run_keys)
    fig    = plt.figure(figsize=(figsize_per_cell[0] * n_cols,
                                 figsize_per_cell[1] * n_rows))
    gs     = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.05, wspace=0.05)
    for ci, key in enumerate(run_keys):
        if key not in som_res: continue
        meta_df, ann_out = som_res[key]
        for ri, (group_label, group_col) in enumerate(groups):
            ax = fig.add_subplot(gs[ri, ci])
            group_portrait(meta_df, ann_out, group_label, group_col, ax=ax,
                           title=str(key) if ri == 0 else None)
            if ci == 0:
                ax.set_ylabel(group_label[:22], fontsize=6)
    fig.suptitle("SOM Portrait Grid — Rows: groups/batches  ·  Cols: top combinations",
                 fontsize=10, y=1.01)
    plt.savefig(f"{OUT_DIR}/portrait_comparison_grid.pdf", bbox_inches="tight")
    plt.show()

plot_master_portrait_grid(som_results, PORTRAIT_GROUPS, list(som_results.keys()))


In [ ]:
# ── 7c. Within-biology batch consistency portraits ──────────────────────────
def plot_within_biology_batch_grid(som_res, diagnosis_group,
                                    diag_col=BIO_COL,
                                    batch_col=BATCH_COL, min_samples=10):
    all_ann = pd.concat([ann_out for _, ann_out in som_res.values()])
    batches = (all_ann.loc[all_ann[diag_col] == diagnosis_group, batch_col]
               .value_counts()
               .pipe(lambda s: s[s >= min_samples])
               .index.tolist())
    keys = list(som_res.keys())
    fig, axs = plt.subplots(len(keys), len(batches),
                             figsize=(1.6 * len(batches), 1.6 * len(keys)),
                             squeeze=False)
    for ri, key in enumerate(keys):
        meta_df, ann_out = som_res[key]
        for ci, batch in enumerate(batches):
            mask = ((ann_out[diag_col] == diagnosis_group) &
                    (ann_out[batch_col] == batch))
            samp = ann_out.index[mask].tolist()
            ax   = axs[ri, ci]
            if ri == 0: ax.set_title(batch[:18], fontsize=5, rotation=30)
            if ci == 0: ax.set_ylabel(str(key)[:18], fontsize=5)
            if samp:
                mean_vec = meta_df[samp].mean(axis=1).values
                grid_sz  = int(np.ceil(np.sqrt(len(mean_vec))))
                portrait = mean_vec[:grid_sz**2].reshape(grid_sz, grid_sz, order="F")
                ax.imshow(portrait, cmap=create_som_colormap(), origin="lower",
                          aspect="auto")
            ax.axis("off")
    fig.suptitle(f"Within-biology batch check: {diagnosis_group}", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/within_biology_{diagnosis_group}.pdf", bbox_inches="tight")
    plt.show()

plot_within_biology_batch_grid(som_results, "Diffuse_Large_B_Cell_Lymphoma")
plot_within_biology_batch_grid(som_results, "Follicular_Lymphoma")


## 8 — Quantitative Summary Table

Merged pre-SOM and post-SOM metrics for all top combinations. Colour-coded: green = low batch effect, red = high.


In [ ]:
from IPython.display import display

final_rows = []
for (strat, imp, meth, pr), (meta_df, ann_out) in som_results.items():
    _key = (strat, imp, meth, pr)
    pre  = combo_df.set_index(
        ["strategy", "imputation", "method", "post_removal"]).loc[_key]
    t    = test_results.get(_key, {})
    row  = {
        "strategy":          strat,
        "imputation":        imp,
        "method":            meth,
        "post_removal":      pr,
        "harshness":         METHODS[meth][1],
        "n_samples":         len(ann_out),
        "pre_batch_r2":      pre["r2_batch"],
        "pre_diag_r2":       pre["r2_diag"],
        "post_batch_r2":     r2_batch(meta_df.T, ann_out, BATCH_COL),
        "post_diag_r2":      r2_batch(meta_df.T, ann_out, BIO_COL),
        "batch_test_passed": t.get("passed", None),
        "same_batch_nn":     t.get("same_batch", None),
        "same_bio_nn":       t.get("same_bio", None),
    }
    final_rows.append(row)

final_df = (pd.DataFrame(final_rows)
              .set_index(["strategy", "imputation", "method", "post_removal"])
              .sort_values("post_batch_r2"))

display(final_df.style
    .background_gradient(cmap="RdYlGn_r",
                         subset=["pre_batch_r2", "post_batch_r2"],
                         axis=0, vmin=0, vmax=1)
    .background_gradient(cmap="RdYlGn",
                         subset=["pre_diag_r2", "post_diag_r2"],
                         axis=0, vmin=0, vmax=0.5)
    .applymap(lambda v: "background-color: #90ee90" if v is True  else
                        "background-color: #ffcccc" if v is False else "",
              subset=["batch_test_passed"]))

final_df.to_csv(f"{OUT_DIR}/final_summary.csv")
print("\nTop 5 combinations by post-SOM batch R²:")
print(final_df.sort_values("post_batch_r2").head(5)
      [["harshness", "n_samples", "pre_batch_r2", "post_batch_r2",
        "batch_test_passed"]].to_string())


## Interpretation

Review the `final_df` table above for the winning combination. Combinations with the lowest `post_batch_r2` best suppress platform-level confounding after SOM. A combination **passes** the batch-dominates-biology test when `same_bio_nn ≥ same_batch_nn` in metagene space — i.e., nearest neighbours are more similar biologically than by batch.

**If no combination passes** the formal batch test with mixed-platform data: platform effects persist because microarray and RNA-seq signals are fundamentally incompatible even after FSQN. The recommended next step is to **restrict SOM analysis to RNA-seq samples only** (Strategy C or a combined C+A filter) and re-run the benchmark within the RNA-seq space.

Key thresholds for interpreting batch R²:
- **< 0.16**: better than or equal to the prior FSQN R baseline
- **< 0.10**: strong harmonization — batch accounts for < 10% of PCA variance
- **> 0.30**: poor harmonization — platform effects still dominate biology
